In [1]:
import os
import pandas as pd
import numpy as np
from urllib.parse import quote_plus
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('MYSQL_USER')}:{quote_plus(os.getenv('MYSQL_PASSWORD'))}"
    f"@{os.getenv('MYSQL_HOST')}/{os.getenv('MYSQL_DB')}"
)
df = pd.read_sql("SELECT * FROM customers_clean", engine)
print(df.shape)
df["MultipleLines"].unique()

(7043, 21)


<StringArray>
['No', 'Yes']
Length: 2, dtype: str

In [2]:
service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

# How many services the customer uses
df["num_services"] = (df[service_cols] == "Yes").sum(axis=1) + (df["InternetService"] != "No")

# Pays automatically (bank transfer or credit card)
df["auto_pay"] = df["PaymentMethod"].str.contains("automatic").astype(int)

# Has online security or tech support
df["has_protection"] = (df[["OnlineSecurity", "TechSupport"]] == "Yes").any(axis=1).astype(int)

# Average spend per month of tenure (new customers use their monthly charge)
df["avg_monthly_spend"] = np.where(df["tenure"] > 0,
                                   df["TotalCharges"] / df["tenure"],
                                   df["MonthlyCharges"])

# Tenure groups
df["tenure_group"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"])

df[["num_services", "auto_pay", "has_protection", "avg_monthly_spend", "tenure_group"]].head()

,num_services,auto_pay,has_protection,avg_monthly_spend,tenure_group
0,2,0,0,29.850000,0-12
1,4,0,1,55.573529,25-48
2,4,0,1,54.075000,0-12
3,4,1,1,40.905556,25-48
4,2,0,0,75.825000,0-12


In [3]:
yes_no = ["Partner", "Dependents", "PhoneService", "PaperlessBilling",
          "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]
for c in yes_no:
    df[c] = df[c].map({"Yes": 1, "No": 0})

df["gender"] = df["gender"].map({"Male": 1, "Female": 0})

df_model = pd.get_dummies(
    df, columns=["InternetService", "Contract", "PaymentMethod", "tenure_group"],
    drop_first=True, dtype=int
)
print(df_model.shape)
print(df_model.isna().sum().sum())   # must be 0
df_model.dtypes

(7043, 32)
0


customerID                                   str
gender                                     int64
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
MultipleLines                              int64
OnlineSecurity                             int64
OnlineBackup                               int64
DeviceProtection                           int64
TechSupport                                int64
StreamingTV                                int64
StreamingMovies                            int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
num_services                               int64
auto_pay                                   int64
has_protection      

In [4]:
df_model.to_sql("customers_features", engine, if_exists="replace", index=False)

7043